## Autonomy - Causal

In [ ]:
## Autonomy - Causal

In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
causal_comparison.py
═════════════════════════════════════════════════════════════════════════════
Causal analysis of Human vs LLM Autonomy Index scoring  —  Layer 4
Companion to complete_comparison.py (Layers 1–3: descriptive, divergence,
Overton pluralistic).  Same apples-to-apples matched data: 30 shared
vignettes, 47 human responses, 2,360 LLM rows (9 models × 10 runs).

ASSUMED CAUSAL DAG
  Vignette ──► [VA, FU, RD, IA] ──► Autonomy Index (AI)
  Rater type ──► domain scoring style ──► AI
  ECI (barriers) ──► AI  (suppressor)
  SPI (support)  ──► AI  (facilitator)

SIX CAUSAL QUESTIONS
  Q1. Variance decomposition
      How much AI variance is explained by vignette, rater type, and their
      interaction?
  Q2. Rater type effect (within-vignette FE)
      Holding vignette constant, what is the causal gap between human and
      LLM scoring?
  Q3. Domain heterogeneity
      Does the Human–LLM gap vary by clinical domain (VA, FU, RD, IA)?
  Q4. Difficulty moderation
      Does the gap widen on harder vignettes (low mean AI)?
  Q5. ECI moderation
      Does the gap depend on external structural constraints?
  Q6. Per-model causal gap
      Which LLM has the smallest causal gap with humans, controlling for
      vignette? Forest plot with 95% CIs.

Usage:
  python causal_comparison.py
  python causal_comparison.py --human <xlsx> --llm <csv> --out <dir>
"""

import os, argparse, warnings, html
from pathlib import Path

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import pearsonr, ttest_rel, ttest_ind
import statsmodels.formula.api as smf
import statsmodels.api         as sm

warnings.filterwarnings("ignore")

# ── constants ─────────────────────────────────────────────────────────────────
ITEM_COLS    = ["VA1","VA2","VA3","VA4","FU1","FU2","FU3",
                "RD1","RD2","RD3","IA1","IA2","IA3"]
DOMAIN_COLS  = ["va_score","fu_score","rd_score","ia_score"]
DOMAIN_SHORT = ["VA","FU","RD","IA"]
DOMAIN_NAMES = {"va_score":"Value Awareness","fu_score":"Factual Understanding",
                "rd_score":"Rational Deliberation","ia_score":"Intentional Action"}
DOMAIN_COLORS= {"VA":"#4C72B0","FU":"#DD8452","RD":"#55A868","IA":"#C44E52"}

H_COLOR   = "#E74C3C"
L_COLOR   = "#2E75B6"
BLUE_DARK = "#1F4E79"
GREEN     = "#27AE60"
GOLD      = "#F39C12"
RED       = "#C0392B"
PURPLE    = "#8E44AD"


def savefig(fig, path):
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  📊 {path}")



In [ ]:
import os; os.getcwd()   # should end in /autonomy_indx_human_comparison

__file__= os.getcwd()
#print(os.getcwd())

# ── Paths (relative to this file's repo — autonomy_indx_human_comparison) ───
#_REPO     = os.path.dirname(os.path.abspath("/Users/taposhduttaroy/workspace/code/autonomy_indx_human_comparison/"))
_REPO     = os.getcwd() # Set _REPO to current working directory
DATA_DIR  = '/content/sample_data' # Fixed syntax error with direct path

#print(DATA_DIR)

MARCH_CSV = os.path.join(DATA_DIR, "scores_march_26_llm_values_50.csv")
JULY_CSV  = os.path.join(DATA_DIR, "scores_july_26_llm_values_50.csv")
HUMAN_CSV = os.path.join(DATA_DIR, "human_july_14_cleaned.csv")
OUT_DIR   = os.path.join(_REPO, "causal")

print(MARCH_CSV)

/content/sample_data/scores_march_26_llm_values_50.csv


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# 0. DATA LOADING + APPLES-TO-APPLES MATCHING
# ═════════════════════════════════════════════════════════════════════════════

def load_human(path):
    """Load human survey data from either xlsx or csv (Qualtrics export)."""
    p = str(path).lower()
    if p.endswith(".csv"):
        raw = pd.read_csv(path)
        # Qualtrics CSV: row 0 = label descriptions, row 1 = JSON metadata
        df = raw.iloc[2:].copy().reset_index(drop=True)
        # Keep only completed responses with a vignette_id
        df = df[df["vignette_id"].notna()]
        df = df[df["Finished"].astype(str).str.lower().isin(["true","1","1.0"])]
        df = df.reset_index(drop=True)
    else:
        raw = pd.read_excel(path)
        df  = raw.iloc[1:].copy().reset_index(drop=True)
    rn  = {"Value Clarity":"VA1","Value Stability":"VA2","Framework Awareness":"VA3",
           "Appreciation":"VA4","Key Facts Recall":"FU1","Risk Comprehension":"FU2",
           "Applicability":"FU3","Coherence":"RD1","Trade-off Reasoning":"RD2",
           "Consistency":"RD3","Intention Strength":"IA1","Planfulness":"IA2",
           "Follow-through F.":"IA3","External Constraint\xa0_1":"ECI",
           "Support Provided_1":"SPI"}
    df = df.rename(columns=rn)
    for c in ITEM_COLS + ["ECI","SPI"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    # ---- PATCH: response-scale decoding, HUMAN data only ----
    # Qualtrics exported six options per item (0-4 plus N/A) as recode
    # values 1..6. stored 6 = N/A (skipna does not catch it, 6 is a number),
    # and stored 1..5 = score 0..4. ECI/SPI are 0-100 sliders: not touched.
    # LLM files already store 0-4 and use load_llm, not this function.
    for _c in ITEM_COLS:
        df[_c] = pd.to_numeric(df[_c], errors="coerce")
        df.loc[df[_c] == 6, _c] = np.nan
        df[_c] = df[_c] - 1
    # ---------------------------------------------------------
    df["va_score"] = df[["VA1","VA2","VA3","VA4"]].mean(axis=1, skipna=True)/4*100
    df["fu_score"] = df[["FU1","FU2","FU3"]].mean(axis=1, skipna=True)/4*100
    df["rd_score"] = df[["RD1","RD2","RD3"]].mean(axis=1, skipna=True)/4*100
    df["ia_score"] = df[["IA1","IA2","IA3"]].mean(axis=1, skipna=True)/4*100
    df["autonomy_index"] = df[DOMAIN_COLS].mean(axis=1, skipna=True)
    df["title_clean"] = df["case_title"].fillna("").apply(
        lambda t: html.unescape(str(t)).strip())
    return df


def load_llm(path):
    df = pd.read_csv(path)
    df = df[df["status"] == "ok"].copy()
    for c in DOMAIN_COLS + ["autonomy_index","ECI","SPI"] + ITEM_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["title_clean"] = df["title"].fillna("").apply(
        lambda t: html.unescape(str(t)).strip())
    return df


def match_and_combine(human, llm):
    """Apples-to-apples: filter to shared vignettes, return combined long table."""
    shared = sorted(set(human["vignette_id"].dropna()) &
                    set(llm["vignette_id"].dropna()))
    h = human[human["vignette_id"].isin(shared)].copy()
    l = llm  [llm  ["vignette_id"].isin(shared)].copy()

    # Unified long format
    h_long = h[["vignette_id","title_clean","autonomy_index",
                "ECI","SPI"] + DOMAIN_COLS].copy()
    h_long["rater_type"]  = "Human"
    h_long["source"]      = "Human"
    h_long["is_human"]    = 1

    l_long = l[["vignette_id","title_clean","autonomy_index",
                "ECI","SPI","model"] + DOMAIN_COLS].copy()
    l_long["rater_type"] = "LLM"
    l_long["source"]     = l_long["model"]
    l_long["is_human"]   = 0
    l_long = l_long.drop(columns=["model"])

    combined = pd.concat([h_long, l_long], ignore_index=True)
    print(f"✓ Matched & combined: {len(combined):,} rows total "
          f"({len(h):,} human, {len(l):,} LLM) across {len(shared)} vignettes")
    return h, l, combined, shared



In [ ]:


# ═════════════════════════════════════════════════════════════════════════════
# Q1. VARIANCE DECOMPOSITION
# ═════════════════════════════════════════════════════════════════════════════

def q1_variance_decomposition(combined):
    """
    Partition AI score variance using nested eta-squared:
      SS_Vignette      = variance explained by vignette mean
      SS_RaterType     = variance explained by rater type (within vignette)
      SS_Interaction   = variance from vignette×rater interaction
      SS_Residual      = remainder
    """
    ai = combined["autonomy_index"].dropna()
    grand = ai.mean()
    ss_total = ((ai - grand)**2).sum()

    df = combined.copy()

    # Vignette effect
    vm = df.groupby("vignette_id")["autonomy_index"].transform("mean")
    ss_vig = ((vm - grand)**2).sum()

    # Rater-type effect (after vignette)
    df["r1"] = df["autonomy_index"] - vm
    rtm = df.groupby("rater_type")["r1"].transform("mean")
    ss_rater = (rtm**2).sum()

    # Interaction (after main effects)
    df["r2"] = df["r1"] - rtm
    intm = df.groupby(["vignette_id","rater_type"])["r2"].transform("mean")
    ss_inter = (intm**2).sum()

    # Residual
    ss_resid = ss_total - ss_vig - ss_rater - ss_inter

    return {
        "ss_total":    ss_total,
        "vignette_pct": 100*ss_vig/ss_total,
        "rater_pct":    100*ss_rater/ss_total,
        "interact_pct": 100*ss_inter/ss_total,
        "residual_pct": 100*ss_resid/ss_total,
    }


# ═════════════════════════════════════════════════════════════════════════════
# Q2. RATER TYPE EFFECT  (within-vignette FE)
# ═════════════════════════════════════════════════════════════════════════════

def q2_rater_type_effect(combined):
    """
    Within-vignette fixed effects: demean AI and is_human by vignette, then
    regress demeaned AI on demeaned is_human. Robust HC3 SE.

    β̂ = causal effect of being a human rater on AI, holding case constant.
    """
    df = combined.dropna(subset=["autonomy_index","is_human"]).copy()
    df["ai_dm"] = df["autonomy_index"] - df.groupby("vignette_id")["autonomy_index"].transform("mean")
    df["is_human_dm"] = df["is_human"] - df.groupby("vignette_id")["is_human"].transform("mean")

    X = sm.add_constant(df[["is_human_dm"]])
    y = df["ai_dm"]
    mod = sm.OLS(y, X).fit(cov_type="HC3")

    beta = mod.params["is_human_dm"]
    se   = mod.bse["is_human_dm"]
    p    = mod.pvalues["is_human_dm"]
    ci   = mod.conf_int().loc["is_human_dm"].values

    return {"beta": beta, "se": se, "p": p,
            "ci_lo": ci[0], "ci_hi": ci[1],
            "N": len(df), "model": mod}


# ═════════════════════════════════════════════════════════════════════════════
# Q3. DOMAIN HETEROGENEITY
# ═════════════════════════════════════════════════════════════════════════════

def q3_domain_heterogeneity(combined):
    """
    For each clinical domain, estimate rater-type effect separately,
    holding vignette constant.
    """
    results = {}
    for col in DOMAIN_COLS:
        df = combined.dropna(subset=[col, "is_human"]).copy()
        df["y_dm"]      = df[col] - df.groupby("vignette_id")[col].transform("mean")
        df["is_h_dm"]   = df["is_human"] - df.groupby("vignette_id")["is_human"].transform("mean")
        X = sm.add_constant(df[["is_h_dm"]])
        y = df["y_dm"]
        mod = sm.OLS(y, X).fit(cov_type="HC3")
        results[col] = {
            "label":  DOMAIN_NAMES[col],
            "beta":   mod.params["is_h_dm"],
            "se":     mod.bse["is_h_dm"],
            "p":      mod.pvalues["is_h_dm"],
            "ci_lo":  mod.conf_int().loc["is_h_dm"].values[0],
            "ci_hi":  mod.conf_int().loc["is_h_dm"].values[1],
            "N":      len(df),
        }
    return results


# ═════════════════════════════════════════════════════════════════════════════
# Q4. DIFFICULTY MODERATION
# ═════════════════════════════════════════════════════════════════════════════

def q4_difficulty_moderation(combined):
    """
    Does the Human-LLM gap depend on vignette difficulty?

    Define difficulty as the LLM-mean AI for that vignette (lower = harder).
    Fit:  AI ~ is_human + difficulty + is_human×difficulty + C(vignette)
    """
    df = combined.dropna(subset=["autonomy_index","is_human"]).copy()

    # Difficulty = LLM-mean AI per vignette (proxy for content difficulty)
    llm_mean = (df[df["is_human"] == 0].groupby("vignette_id")["autonomy_index"]
                .mean().rename("difficulty"))
    df = df.merge(llm_mean.reset_index(), on="vignette_id", how="left")
    df["difficulty_c"] = df["difficulty"] - df["difficulty"].mean()
    df["is_human_c"]   = df["is_human"]

    # Interaction model (demean AI by vignette to absorb vignette FE)
    df["ai_dm"] = df["autonomy_index"] - df.groupby("vignette_id")["autonomy_index"].transform("mean")
    df["is_h_dm"] = df["is_human_c"] - df.groupby("vignette_id")["is_human_c"].transform("mean")
    # difficulty is constant within vignette, so dm → 0. We interact raw is_human_dm with difficulty_c
    df["inter"] = df["is_h_dm"] * df["difficulty_c"]

    X = sm.add_constant(df[["is_h_dm","inter"]])
    y = df["ai_dm"]
    mod = sm.OLS(y, X).fit(cov_type="HC3")

    return {
        "main_effect":     mod.params["is_h_dm"],
        "interaction":     mod.params["inter"],
        "p_interaction":   mod.pvalues["inter"],
        "ci_interaction":  mod.conf_int().loc["inter"].values.tolist(),
        "mean_difficulty": df["difficulty"].mean(),
        "model":           mod,
        "df":              df,
    }


# ═════════════════════════════════════════════════════════════════════════════
# Q5. ECI MODERATION  (structural barriers)
# ═════════════════════════════════════════════════════════════════════════════

def q5_eci_moderation(combined):
    """
    Does the Human-LLM gap depend on external constraint (ECI)?

    Fit: AI ~ is_human + ECI + is_human × ECI + C(vignette)
    """
    df = combined.dropna(subset=["autonomy_index","is_human","ECI"]).copy()
    df["ai_dm"] = df["autonomy_index"] - df.groupby("vignette_id")["autonomy_index"].transform("mean")
    df["eci_dm"] = df["ECI"] - df.groupby("vignette_id")["ECI"].transform("mean")
    df["is_h_dm"] = df["is_human"] - df.groupby("vignette_id")["is_human"].transform("mean")
    df["inter_heci"] = df["is_h_dm"] * df["eci_dm"]

    X = sm.add_constant(df[["is_h_dm","eci_dm","inter_heci"]])
    y = df["ai_dm"]
    mod = sm.OLS(y, X).fit(cov_type="HC3")

    # Same for SPI
    df2 = combined.dropna(subset=["autonomy_index","is_human","SPI"]).copy()
    df2["ai_dm"]  = df2["autonomy_index"] - df2.groupby("vignette_id")["autonomy_index"].transform("mean")
    df2["spi_dm"] = df2["SPI"]            - df2.groupby("vignette_id")["SPI"].transform("mean")
    df2["is_h_dm"] = df2["is_human"] - df2.groupby("vignette_id")["is_human"].transform("mean")
    df2["inter_hspi"] = df2["is_h_dm"] * df2["spi_dm"]
    X2 = sm.add_constant(df2[["is_h_dm","spi_dm","inter_hspi"]])
    y2 = df2["ai_dm"]
    mod2 = sm.OLS(y2, X2).fit(cov_type="HC3")

    return {
        "eci": {
            "main_h":    mod.params["is_h_dm"],
            "main_eci":  mod.params["eci_dm"],
            "inter":     mod.params["inter_heci"],
            "p_inter":   mod.pvalues["inter_heci"],
            "ci_inter":  mod.conf_int().loc["inter_heci"].values.tolist(),
        },
        "spi": {
            "main_h":    mod2.params["is_h_dm"],
            "main_spi":  mod2.params["spi_dm"],
            "inter":     mod2.params["inter_hspi"],
            "p_inter":   mod2.pvalues["inter_hspi"],
            "ci_inter":  mod2.conf_int().loc["inter_hspi"].values.tolist(),
        },
    }


# ═════════════════════════════════════════════════════════════════════════════
# Q6. PER-MODEL CAUSAL GAP  (each LLM vs human, vignette-adjusted)
# ═════════════════════════════════════════════════════════════════════════════

def q6_per_model_gap(combined, llm):
    """
    For each LLM model separately, estimate its causal gap from humans on the
    same vignettes, using within-vignette demeaning.

    β̂_m = Mean_m(AI) - Mean_Human(AI), adjusted for vignette content.
    """
    results = {}
    base = combined.dropna(subset=["autonomy_index","is_human"])
    human_rows = base[base["is_human"] == 1]
    for model_name in sorted(llm["model"].unique()):
        model_rows = base[base["source"] == model_name]
        sub = pd.concat([human_rows, model_rows], ignore_index=True)
        sub["ai_dm"]    = sub["autonomy_index"] - sub.groupby("vignette_id")["autonomy_index"].transform("mean")
        sub["is_h_dm"]  = sub["is_human"]       - sub.groupby("vignette_id")["is_human"].transform("mean")

        X = sm.add_constant(sub[["is_h_dm"]])
        y = sub["ai_dm"]
        m = sm.OLS(y, X).fit(cov_type="HC3")
        # We want LLM effect vs human → negate the human coefficient
        llm_gap = -m.params["is_h_dm"]   # positive = LLM > human
        se      = m.bse["is_h_dm"]
        ci      = m.conf_int().loc["is_h_dm"].values
        results[model_name] = {
            "gap":       llm_gap,
            "se":        se,
            "p":         m.pvalues["is_h_dm"],
            "ci_lo":     -ci[1],   # flip sign for LLM-vs-human interpretation
            "ci_hi":     -ci[0],
            "N":         len(sub),
        }
    return results


# ═════════════════════════════════════════════════════════════════════════════
# FIGURES
# ═════════════════════════════════════════════════════════════════════════════

# ── Fig CC1: Variance decomposition + rater effect ────────────────────────
def fig_cc1_variance(q1, q2, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # Left: pie
    ax = axes[0]
    parts = {
        "Vignette\n(case content)":  q1["vignette_pct"],
        "Rater type\n(Human vs LLM)":q1["rater_pct"],
        "Interaction\n(case × rater)": q1["interact_pct"],
        "Residual\n(unexplained)":   q1["residual_pct"],
    }
    colors = ["#1F4E79", "#E74C3C", "#F39C12", "#BDC3C7"]
    wedges, texts, autos = ax.pie(
        list(parts.values()), labels=list(parts.keys()),
        colors=colors, autopct="%1.1f%%", startangle=140,
        pctdistance=0.72, wedgeprops=dict(edgecolor="white", linewidth=2)
    )
    for t in autos:
        t.set_fontsize(10); t.set_fontweight("bold")
    ax.set_title("Q1 — AI Score Variance Partition",
                  fontsize=12, fontweight="bold")

    # Right: rater type effect — forest style
    ax = axes[1]
    beta, se, p, lo, hi = q2["beta"], q2["se"], q2["p"], q2["ci_lo"], q2["ci_hi"]
    ax.errorbar([beta], [0], xerr=[[beta-lo],[hi-beta]], fmt="o",
                markersize=16, color=H_COLOR, capsize=10, capthick=2,
                markeredgecolor="black", markeredgewidth=1.5, zorder=3)
    ax.axvline(0, color="black", lw=1.5, linestyle="--", alpha=0.7)
    ax.fill_betweenx([-0.4,0.4], lo, hi, alpha=0.15, color=H_COLOR)
    ax.set_yticks([0]); ax.set_yticklabels(["Human − LLM"], fontsize=11)
    ax.set_xlabel("Causal effect on AI score (pts)", fontsize=11)
    sig = "p < 0.001" if p < 0.001 else f"p = {p:.4f}"
    ax.set_title(f"Q2 — Within-Vignette Rater Effect\nβ = {beta:+.2f}  [95% CI {lo:+.2f}, {hi:+.2f}]  {sig}",
                 fontsize=11, fontweight="bold")
    ax.text(beta, 0.2, f"{beta:+.2f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_xlim(min(lo-3, -1), max(hi+3, 1))
    ax.grid(axis="x", alpha=0.3)

    # Interpretation
    direction = "higher" if beta > 0 else "lower"
    ax.text(0.02, 0.02,
            f"Interpretation: holding vignette content constant,\n"
            f"human raters score the Autonomy Index {abs(beta):.2f} pts {direction}\n"
            f"than LLMs (on average).",
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle="round", facecolor="#FEF9E7", alpha=0.85))

    fig.suptitle("Q1–Q2 — Variance Decomposition & Causal Effect of Rater Type",
                 fontsize=13, fontweight="bold", y=1.02, color=BLUE_DARK)
    plt.tight_layout()
    savefig(fig, os.path.join(out_dir, "figCC1_variance_rater.png"))


# ── Fig CC2: Domain heterogeneity ─────────────────────────────────────────
def fig_cc2_domain_heterogeneity(q3, out_dir):
    fig, ax = plt.subplots(figsize=(11, 5.5))

    domains = DOMAIN_COLS
    labels  = [DOMAIN_NAMES[c] for c in domains]
    betas   = [q3[c]["beta"]  for c in domains]
    ses     = [q3[c]["se"]    for c in domains]
    los     = [q3[c]["ci_lo"] for c in domains]
    his     = [q3[c]["ci_hi"] for c in domains]
    pvals   = [q3[c]["p"]     for c in domains]
    colors  = [DOMAIN_COLORS[s] for s in DOMAIN_SHORT]

    y = np.arange(len(domains))
    ax.errorbar(betas, y, xerr=[[b-l for b,l in zip(betas,los)],
                                 [h-b for b,h in zip(betas,his)]],
                 fmt="o", markersize=14, capsize=8, capthick=2,
                 markeredgecolor="black", markeredgewidth=1.5,
                 ecolor="black", zorder=3,
                 markerfacecolor="white")
    # Colour markers individually
    for xi, yi, color in zip(betas, y, colors):
        ax.plot(xi, yi, "o", markersize=14, color=color,
                markeredgecolor="black", markeredgewidth=1.5, zorder=4)

    ax.axvline(0, color="black", lw=1.5, linestyle="--", alpha=0.7)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel("Causal Human-LLM gap (pts, vignette-adjusted)", fontsize=11)
    ax.set_title("Q3 — Heterogeneous Rater-Type Effect by Clinical Domain\n"
                 "(positive = humans score this domain higher than LLMs)",
                 fontsize=12, fontweight="bold")

    for yi, (b, p) in enumerate(zip(betas, pvals)):
        star = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "n.s."
        ax.text(b + 0.5, yi + 0.2, f"β = {b:+.2f}\n{star}",
                 fontsize=9, fontweight="bold")

    # Annotate the interpretation band
    ax.axvspan(min(los)-2, 0, alpha=0.05, color=L_COLOR)
    ax.axvspan(0, max(his)+2, alpha=0.05, color=H_COLOR)
    ax.text(-0.5, -0.8, "LLMs score higher", color=L_COLOR, fontsize=9, ha="right")
    ax.text( 0.5, -0.8, "Humans score higher", color=H_COLOR, fontsize=9)

    plt.tight_layout()
    savefig(fig, os.path.join(out_dir, "figCC2_domain_heterogeneity.png"))


# ── Fig CC3: Difficulty moderation ────────────────────────────────────────
def fig_cc3_difficulty(q4, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # Left: interaction plot
    ax = axes[0]
    df   = q4["df"]
    diff = df.groupby("vignette_id")["difficulty"].first().sort_values()

    # Per-vignette human and LLM means
    per_vig = (df.groupby(["vignette_id","rater_type"])["autonomy_index"]
               .mean().unstack("rater_type"))
    per_vig = per_vig.join(diff, how="left").sort_values("difficulty")

    ax.scatter(per_vig["difficulty"], per_vig["Human"],
               s=80, color=H_COLOR, alpha=0.75, edgecolors="white",
               label="Human", zorder=3)
    ax.scatter(per_vig["difficulty"], per_vig["LLM"],
               s=80, color=L_COLOR, alpha=0.75, edgecolors="white",
               label="LLM", zorder=3)

    # Trend lines
    m_h, b_h = np.polyfit(per_vig["difficulty"].dropna(), per_vig["Human"].dropna(), 1)
    m_l, b_l = np.polyfit(per_vig["difficulty"].dropna(), per_vig["LLM"].dropna(),   1)
    x_r = np.linspace(per_vig["difficulty"].min(), per_vig["difficulty"].max(), 100)
    ax.plot(x_r, m_h*x_r + b_h, "-", color=H_COLOR, lw=2, alpha=0.7,
            label=f"Human slope = {m_h:.2f}")
    ax.plot(x_r, m_l*x_r + b_l, "-", color=L_COLOR, lw=2, alpha=0.7,
            label=f"LLM slope = {m_l:.2f}")

    ax.set_xlabel("Vignette difficulty (LLM mean AI — lower = harder)", fontsize=11)
    ax.set_ylabel("Human or LLM mean AI", fontsize=11)
    ax.set_title("Q4 — Human vs LLM by Vignette Difficulty\n"
                 "Parallel slopes = no interaction; divergent = gap moderated",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)

    # Right: interaction coefficient with CI
    ax = axes[1]
    inter = q4["interaction"]
    lo, hi = q4["ci_interaction"]
    ax.errorbar([inter], [0], xerr=[[inter-lo],[hi-inter]], fmt="o",
                markersize=18, color=PURPLE, capsize=10, capthick=2,
                markeredgecolor="black", markeredgewidth=1.5, zorder=3)
    ax.axvline(0, color="black", lw=1.5, linestyle="--", alpha=0.7)
    ax.fill_betweenx([-0.4, 0.4], lo, hi, alpha=0.15, color=PURPLE)
    ax.set_yticks([0])
    ax.set_yticklabels(["is_human × difficulty"], fontsize=10)
    sig = "p < 0.001" if q4["p_interaction"] < 0.001 else f"p = {q4['p_interaction']:.4f}"
    ax.set_title(f"Interaction coefficient\nβ = {inter:+.4f}  {sig}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Moderation effect (units: AI per unit-difficulty)", fontsize=10)
    ax.grid(axis="x", alpha=0.3)

    interp = ("Gap widens on harder cases" if inter > 0.01 else
               "Gap narrows on harder cases" if inter < -0.01 else
               "Gap is roughly constant across difficulty")
    ax.text(0.02, 0.02, f"Interpretation: {interp}",
            transform=ax.transAxes, fontsize=9.5,
            bbox=dict(boxstyle="round", facecolor="#EBF5FB", alpha=0.85))

    fig.suptitle("Q4 — Does Vignette Difficulty Moderate the Human-LLM Gap?",
                 fontsize=13, fontweight="bold", y=1.02, color=BLUE_DARK)
    plt.tight_layout()
    savefig(fig, os.path.join(out_dir, "figCC3_difficulty_moderation.png"))


# ── Fig CC4: ECI / SPI moderation ─────────────────────────────────────────
def fig_cc4_eci_spi(q5, combined, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    for ax, modifier, color_mod, label in [
        (axes[0], "ECI", "#922B21", "External Constraints (ECI)"),
        (axes[1], "SPI", "#1E8449", "Support Provided (SPI)"),
    ]:
        # Terciles of modifier
        df = combined.dropna(subset=["autonomy_index","is_human",modifier]).copy()
        df["tercile"] = pd.qcut(df[modifier], q=3, labels=["Low","Med","High"])
        mean_ai = df.groupby(["tercile","rater_type"])["autonomy_index"].mean().unstack("rater_type")

        x = np.arange(3); w = 0.35
        ax.bar(x-w/2, mean_ai["Human"], width=w, color=H_COLOR, alpha=0.85,
                label="Human", edgecolor="white")
        ax.bar(x+w/2, mean_ai["LLM"], width=w, color=L_COLOR, alpha=0.85,
                label="LLM", edgecolor="white")

        # Gap annotations
        for xi in x:
            h_v = mean_ai["Human"].iloc[xi]
            l_v = mean_ai["LLM"].iloc[xi]
            gap = h_v - l_v
            ax.text(xi, max(h_v, l_v) + 3, f"Δ = {gap:+.1f}",
                     ha="center", fontsize=10, fontweight="bold",
                     color=H_COLOR if gap > 0 else L_COLOR)

        ax.set_xticks(x)
        ax.set_xticklabels([f"Low\n{modifier}", f"Med\n{modifier}", f"High\n{modifier}"],
                            fontsize=10)
        ax.set_ylabel("Mean AI Score", fontsize=11)
        sub = q5[modifier.lower()]
        p_ = sub["p_inter"]
        sig = "p < 0.001" if p_ < 0.001 else f"p = {p_:.3f}"
        ax.set_title(f"{label} Moderation\nInteraction β = {sub['inter']:+.4f}  {sig}",
                     fontsize=11, fontweight="bold", color=color_mod)
        ax.legend(fontsize=10)
        ax.set_ylim(0, 85)

    fig.suptitle("Q5 — Does the Human-LLM Gap Depend on Structural Context?",
                 fontsize=13, fontweight="bold", y=1.02, color=BLUE_DARK)
    plt.tight_layout()
    savefig(fig, os.path.join(out_dir, "figCC4_eci_spi_moderation.png"))


# ── Fig CC5: Per-model forest plot ────────────────────────────────────────
def fig_cc5_forest_models(q6, out_dir):
    df = pd.DataFrame(q6).T.sort_values("gap")
    fig, ax = plt.subplots(figsize=(12, 6.5))

    y = np.arange(len(df))
    colors = ["#27AE60" if abs(g) < 5 else
              ("#F39C12" if abs(g) < 15 else "#C0392B")
              for g in df["gap"]]

    for yi, (model, row) in enumerate(df.iterrows()):
        ax.errorbar(row["gap"], yi,
                    xerr=[[row["gap"]-row["ci_lo"]], [row["ci_hi"]-row["gap"]]],
                    fmt="o", markersize=13, color=colors[yi],
                    capsize=6, capthick=2,
                    markeredgecolor="black", markeredgewidth=1.2)

    ax.axvline(0, color="black", lw=1.5, linestyle="--", alpha=0.7,
               label="No gap (perfect human-match)")
    ax.set_yticks(y)
    ax.set_yticklabels([m.replace(" Preview","").replace(" Flash Lite","")
                         for m in df.index], fontsize=11)
    ax.set_xlabel("LLM − Human causal gap (AI score pts, vignette-adjusted)",
                  fontsize=11)
    ax.set_title("Q6 — Per-Model Causal Gap vs Human Raters\n"
                 "(negative = LLM scores lower than humans, vignette-adjusted)",
                 fontsize=12, fontweight="bold")

    # Annotate with values and stars
    for yi, (model, row) in enumerate(df.iterrows()):
        star = ("***" if row["p"] < 0.001 else
                 "**"  if row["p"] < 0.01  else
                 "*"   if row["p"] < 0.05  else "n.s.")
        ax.text(row["ci_hi"] + 1, yi,
                f"Δ = {row['gap']:+.2f}  [{row['ci_lo']:+.1f}, {row['ci_hi']:+.1f}]  {star}",
                va="center", fontsize=8.5)

    leg = [mpatches.Patch(color="#27AE60", label="|Δ| < 5  (close to human)"),
           mpatches.Patch(color="#F39C12", label="5 ≤ |Δ| < 15  (moderate gap)"),
           mpatches.Patch(color="#C0392B", label="|Δ| ≥ 15  (large gap)")]
    ax.legend(handles=leg, fontsize=9, loc="lower right", title="Gap size")

    plt.tight_layout()
    savefig(fig, os.path.join(out_dir, "figCC5_per_model_forest.png"))


# ── Fig CC6: Dashboard ────────────────────────────────────────────────────
def fig_cc6_dashboard(q1, q2, q3, q4, q5, q6, out_dir):
    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.38)

    # A: Variance pie
    ax = fig.add_subplot(gs[0, 0])
    parts = {"Vignette":q1["vignette_pct"], "Rater":q1["rater_pct"],
              "Interact":q1["interact_pct"], "Residual":q1["residual_pct"]}
    colors = ["#1F4E79","#E74C3C","#F39C12","#BDC3C7"]
    _, _, auto = ax.pie(list(parts.values()), labels=list(parts.keys()),
                         colors=colors, autopct="%1.1f%%", startangle=140,
                         pctdistance=0.72,
                         wedgeprops=dict(edgecolor="white", linewidth=2))
    for t in auto: t.set_fontsize(9); t.set_fontweight("bold")
    ax.set_title("Q1 — Variance\nDecomposition", fontsize=10, fontweight="bold")

    # B: Rater type effect
    ax = fig.add_subplot(gs[0, 1])
    ax.errorbar([q2["beta"]], [0],
                xerr=[[q2["beta"]-q2["ci_lo"]], [q2["ci_hi"]-q2["beta"]]],
                fmt="o", markersize=18, color=H_COLOR, capsize=10,
                markeredgecolor="black", markeredgewidth=1.5)
    ax.axvline(0, color="black", lw=1.5, linestyle="--", alpha=0.7)
    ax.set_yticks([0]); ax.set_yticklabels(["Human - LLM"], fontsize=10)
    ax.set_xlabel("β (AI pts)", fontsize=9)
    ax.set_title(f"Q2 — Rater Effect\nβ={q2['beta']:+.2f}", fontsize=10, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)

    # C: Domain heterogeneity
    ax = fig.add_subplot(gs[0, 2])
    labels  = DOMAIN_SHORT
    betas   = [q3[c]["beta"]  for c in DOMAIN_COLS]
    los     = [q3[c]["ci_lo"] for c in DOMAIN_COLS]
    his     = [q3[c]["ci_hi"] for c in DOMAIN_COLS]
    colors  = [DOMAIN_COLORS[s] for s in labels]
    y = np.arange(len(labels))
    for yi, (b, l, h, c_) in enumerate(zip(betas, los, his, colors)):
        ax.errorbar([b], [yi], xerr=[[b-l],[h-b]], fmt="o",
                    markersize=11, color=c_, capsize=5,
                    markeredgecolor="black", markeredgewidth=1.2)
    ax.axvline(0, color="black", lw=1.2, linestyle="--", alpha=0.7)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel("H-L gap per domain", fontsize=9)
    ax.set_title("Q3 — Domain-Level\nHeterogeneity", fontsize=10, fontweight="bold")

    # D: Per-model forest
    ax = fig.add_subplot(gs[1, :2])
    df_q6 = pd.DataFrame(q6).T.sort_values("gap")
    y = np.arange(len(df_q6))
    for yi, (model, row) in enumerate(df_q6.iterrows()):
        color = ("#27AE60" if abs(row["gap"])<5 else
                 "#F39C12" if abs(row["gap"])<15 else "#C0392B")
        ax.errorbar(row["gap"], yi,
                    xerr=[[row["gap"]-row["ci_lo"]],[row["ci_hi"]-row["gap"]]],
                    fmt="o", markersize=11, color=color, capsize=5,
                    markeredgecolor="black", markeredgewidth=1.2)
    ax.axvline(0, color="black", lw=1.5, linestyle="--", alpha=0.7)
    ax.set_yticks(y)
    ax.set_yticklabels([m.replace(" Preview","").replace(" Flash Lite","")
                         for m in df_q6.index], fontsize=10)
    ax.set_xlabel("LLM − Human causal gap (AI pts)", fontsize=10)
    ax.set_title("Q6 — Per-Model Causal Gap vs Human\n(closer to 0 = more human-like)",
                 fontsize=11, fontweight="bold")
    for yi, (model, row) in enumerate(df_q6.iterrows()):
        ax.text(row["ci_hi"]+1, yi, f"{row['gap']:+.1f}", va="center", fontsize=8)

    # E: ECI / SPI interactions
    ax = fig.add_subplot(gs[1, 2])
    ax.axis("off")
    lines = [
        ("CAUSAL FINDINGS SUMMARY", "black", 12, True),
        ("", "black", 9, False),
        (f"Q1 — Vignette explains {q1['vignette_pct']:.1f}% of AI variance", "#333", 10, False),
        (f"     Rater type explains {q1['rater_pct']:.1f}%", "#333", 10, False),
        ("", "black", 9, False),
        (f"Q2 — Within-vignette gap:  β = {q2['beta']:+.2f}  p = {q2['p']:.4f}", H_COLOR, 10, True),
        ("", "black", 9, False),
        ("Q3 — Largest domain gap:", "#333", 10, True),
    ]
    domain_max = max(DOMAIN_COLS, key=lambda c: abs(q3[c]["beta"]))
    lines += [
        (f"     {DOMAIN_NAMES[domain_max]}: β={q3[domain_max]['beta']:+.2f}", "#333", 10, False),
        ("", "black", 9, False),
        ("Q4 — Difficulty interaction:", "#333", 10, True),
        (f"     β_inter = {q4['interaction']:+.4f}  p = {q4['p_interaction']:.3f}", "#333", 10, False),
        ("", "black", 9, False),
        ("Q5 — ECI moderation:", "#333", 10, True),
        (f"     β_inter = {q5['eci']['inter']:+.4f}  p = {q5['eci']['p_inter']:.3f}", "#333", 10, False),
        ("", "black", 9, False),
        ("Q6 — Best model match:", "#333", 10, True),
    ]
    best_model = df_q6.iloc[(df_q6["gap"].abs()).argmin()]
    lines += [
        (f"     {best_model.name[:30]}", "#333", 9, False),
        (f"     gap = {best_model['gap']:+.2f}", "#333", 10, False),
    ]
    y0 = 0.98
    for line, color, size, bold in lines:
        ax.text(0.02, y0, line, transform=ax.transAxes,
                fontsize=size, color=color, fontweight="bold" if bold else "normal",
                va="top")
        y0 -= 0.055

    fig.suptitle("Causal Analysis Dashboard — Human vs LLM Autonomy Index Scoring",
                 fontsize=14, fontweight="bold", y=1.01, color=BLUE_DARK)
    savefig(fig, os.path.join(out_dir, "figCC6_dashboard.png"))





In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# REPORT + CSV
# ═════════════════════════════════════════════════════════════════════════════

def save_and_report(q1, q2, q3, q4, q5, q6, out_dir):
    sep = "─" * 72
    print(f"\n{'═'*72}")
    print("  CAUSAL COMPARISON REPORT — HUMAN vs LLM AUTONOMY INDEX")
    print(f"{'═'*72}")

    print(f"\n{sep}")
    print("Q1 — VARIANCE DECOMPOSITION")
    for label, key in [("Vignette (case content)","vignette_pct"),
                        ("Rater type (Human/LLM)", "rater_pct"),
                        ("Interaction (case × rater)","interact_pct"),
                        ("Residual (unexplained)","residual_pct")]:
        pct = q1[key]
        bar = "█" * int(pct/2)
        print(f"  {label:<32}  {pct:5.1f}%  {bar}")

    print(f"\n{sep}")
    print("Q2 — CAUSAL EFFECT OF RATER TYPE (within-vignette)")
    sig = "p < 0.001" if q2["p"] < 0.001 else f"p = {q2['p']:.4f}"
    print(f"  β = {q2['beta']:+.3f}  SE = {q2['se']:.3f}  [95% CI {q2['ci_lo']:+.2f}, {q2['ci_hi']:+.2f}]  {sig}")
    direction = "HIGHER" if q2["beta"] > 0 else "LOWER"
    print(f"  → Human raters score {abs(q2['beta']):.2f} pts {direction} than LLMs, same vignette")
    print(f"  (N = {q2['N']:,} observations)")

    print(f"\n{sep}")
    print("Q3 — DOMAIN HETEROGENEITY (per-domain rater effect)")
    print(f"  {'Domain':<25}  {'β':>8}  {'95% CI':>20}  sig")
    for col in DOMAIN_COLS:
        r = q3[col]
        star = "***" if r["p"]<0.001 else "**" if r["p"]<0.01 else "*" if r["p"]<0.05 else "n.s."
        print(f"  {r['label']:<25}  {r['beta']:>+8.2f}  "
              f"[{r['ci_lo']:+6.2f}, {r['ci_hi']:+6.2f}]  {star}")

    print(f"\n{sep}")
    print("Q4 — DIFFICULTY MODERATION (gap × vignette difficulty)")
    p_str = "p < 0.001" if q4["p_interaction"] < 0.001 else f"p = {q4['p_interaction']:.4f}"
    print(f"  Interaction β = {q4['interaction']:+.4f}  "
          f"CI [{q4['ci_interaction'][0]:+.4f}, {q4['ci_interaction'][1]:+.4f}]  {p_str}")
    if q4["p_interaction"] < 0.05:
        direction = "widens" if q4["interaction"] > 0 else "narrows"
        print(f"  → Gap SIGNIFICANTLY {direction} with vignette difficulty")
    else:
        print(f"  → No significant moderation by difficulty (gap is stable across cases)")

    print(f"\n{sep}")
    print("Q5 — CONTEXTUAL MODIFIERS (ECI, SPI)")
    for mod_name, data in [("ECI", q5["eci"]), ("SPI", q5["spi"])]:
        p_str = "p < 0.001" if data["p_inter"] < 0.001 else f"p = {data['p_inter']:.4f}"
        print(f"  {mod_name} × is_human interaction: β = {data['inter']:+.4f}  "
              f"CI [{data['ci_inter'][0]:+.4f}, {data['ci_inter'][1]:+.4f}]  {p_str}")

    print(f"\n{sep}")
    print("Q6 — PER-MODEL CAUSAL GAP (LLM − Human, vignette-adjusted)")
    df_q6 = pd.DataFrame(q6).T.sort_values("gap", key=abs)
    for model, row in df_q6.iterrows():
        short = model.replace(" Preview","").replace(" Flash Lite","")
        if pd.isna(row["gap"]):
            print(f"  {short:<38}  Δ =    n/a  (insufficient overlap / degenerate fit)")
            continue
        star = ("***" if row["p"]<0.001 else "**" if row["p"]<0.01 else
                "*"   if row["p"]<0.05  else "n.s.")
        bar = "▲" * int(abs(row["gap"])/2) if row["gap"] > 0 else "▼" * int(abs(row["gap"])/2)
        print(f"  {short:<38}  Δ = {row['gap']:+6.2f}  "
              f"[{row['ci_lo']:+6.2f}, {row['ci_hi']:+6.2f}]  {star}  {bar}")

    print(f"\n{'═'*72}")

    # ── CSVs ────────────────────────────────────────────────────────────────
    # Q3 domain table
    rows = []
    for col, r in q3.items():
        rows.append({"domain": r["label"], "beta": r["beta"], "se": r["se"],
                      "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"], "p": r["p"]})
    pd.DataFrame(rows).round(4).to_csv(
        os.path.join(out_dir, "causal_domain_heterogeneity.csv"), index=False)

    # Q6 model table
    df_q6_out = df_q6.copy().reset_index().rename(columns={"index":"model"})
    df_q6_out.round(4).to_csv(
        os.path.join(out_dir, "causal_per_model_gap.csv"), index=False)

    # Master summary
    summary_rows = [
        {"question":"Q1_vignette_pct",    "value": round(q1["vignette_pct"], 3)},
        {"question":"Q1_rater_pct",       "value": round(q1["rater_pct"],    3)},
        {"question":"Q1_interact_pct",    "value": round(q1["interact_pct"], 3)},
        {"question":"Q1_residual_pct",    "value": round(q1["residual_pct"], 3)},
        {"question":"Q2_rater_beta",      "value": round(q2["beta"],  4)},
        {"question":"Q2_rater_p",         "value": round(q2["p"],     4)},
        {"question":"Q4_interaction_beta","value": round(q4["interaction"], 4)},
        {"question":"Q4_interaction_p",   "value": round(q4["p_interaction"], 4)},
        {"question":"Q5_ECI_inter_beta",  "value": round(q5["eci"]["inter"],  4)},
        {"question":"Q5_ECI_inter_p",     "value": round(q5["eci"]["p_inter"], 4)},
        {"question":"Q5_SPI_inter_beta",  "value": round(q5["spi"]["inter"],  4)},
        {"question":"Q5_SPI_inter_p",     "value": round(q5["spi"]["p_inter"], 4)},
    ]
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(out_dir, "causal_summary.csv"), index=False)

    print(f"\nCSVs saved:")
    print(f"  • {out_dir}/causal_summary.csv")
    print(f"  • {out_dir}/causal_domain_heterogeneity.csv")
    print(f"  • {out_dir}/causal_per_model_gap.csv")



In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# MAIN
# ═════════════════════════════════════════════════════════════════════════════

def main(human_path, llm_path, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    print("Loading data...")
    human_raw = load_human(human_path)
    llm_raw   = load_llm(llm_path)
    print(f"  Human raw: {len(human_raw)} responses, {human_raw['vignette_id'].nunique()} vignettes")
    print(f"  LLM raw:   {len(llm_raw):,} rows, {llm_raw['vignette_id'].nunique()} vignettes")

    print("\nApples-to-apples matching...")
    human, llm, combined, shared = match_and_combine(human_raw, llm_raw)

    print("\n[Q1] Variance decomposition...")
    q1 = q1_variance_decomposition(combined)
    print(f"      Vignette={q1['vignette_pct']:.1f}%  Rater={q1['rater_pct']:.1f}%")

    print("[Q2] Rater type effect (within-vignette FE)...")
    q2 = q2_rater_type_effect(combined)
    print(f"      β = {q2['beta']:+.3f}  p = {q2['p']:.4f}")

    print("[Q3] Domain heterogeneity...")
    q3 = q3_domain_heterogeneity(combined)

    print("[Q4] Difficulty moderation...")
    q4 = q4_difficulty_moderation(combined)
    print(f"      Interaction β = {q4['interaction']:+.4f}  p = {q4['p_interaction']:.4f}")

    print("[Q5] ECI/SPI moderation...")
    q5 = q5_eci_moderation(combined)

    print("[Q6] Per-model causal gap...")
    q6 = q6_per_model_gap(combined, llm)

    print("\nGenerating causal figures:")
    fig_cc1_variance(q1, q2, out_dir)
    fig_cc2_domain_heterogeneity(q3, out_dir)
    fig_cc3_difficulty(q4, out_dir)
    fig_cc4_eci_spi(q5, combined, out_dir)
    fig_cc5_forest_models(q6, out_dir)
    fig_cc6_dashboard(q1, q2, q3, q4, q5, q6, out_dir)

    save_and_report(q1, q2, q3, q4, q5, q6, out_dir)

    return {"q1":q1,"q2":q2,"q3":q3,"q4":q4,"q5":q5,"q6":q6,
            "human":human,"llm":llm,"combined":combined}


if __name__ == "__main__":
    # Removed argparse as it causes issues in Colab when executed as a cell.
    # Using predefined global paths instead.
    main(HUMAN_CSV, JULY_CSV, OUT_DIR)


Loading data...
  Human raw: 174 responses, 51 vignettes
  LLM raw:   3,431 rows, 50 vignettes

Apples-to-apples matching...
✓ Matched & combined: 3,604 rows total (173 human, 3,431 LLM) across 50 vignettes

[Q1] Variance decomposition...
      Vignette=67.6%  Rater=15.1%
[Q2] Rater type effect (within-vignette FE)...
      β = +55.637  p = 0.0000
[Q3] Domain heterogeneity...
[Q4] Difficulty moderation...
      Interaction β = -0.3558  p = 0.0000
[Q5] ECI/SPI moderation...
[Q6] Per-model causal gap...

Generating causal figures:
  📊 /content/causal/figCC1_variance_rater.png
  📊 /content/causal/figCC2_domain_heterogeneity.png
  📊 /content/causal/figCC3_difficulty_moderation.png
  📊 /content/causal/figCC4_eci_spi_moderation.png
  📊 /content/causal/figCC5_per_model_forest.png
  📊 /content/causal/figCC6_dashboard.png

════════════════════════════════════════════════════════════════════════
  CAUSAL COMPARISON REPORT — HUMAN vs LLM AUTONOMY INDEX
════════════════════════════════════════════

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Call main function to get the combined DataFrame
results = main(HUMAN_CSV, JULY_CSV, OUT_DIR)
combined = results['combined']

fig, ax = plt.subplots(figsize=(10, 6))

sns.kdeplot(data=combined[combined['is_human'] == 1], x='autonomy_index',
            fill=True, color=H_COLOR, label='Human', ax=ax)
sns.kdeplot(data=combined[combined['is_human'] == 0], x='autonomy_index',
            fill=True, color=L_COLOR, label='LLM', ax=ax)

ax.set_title('Distribution of Autonomy Index: Human vs. LLM', fontsize=14)
ax.set_xlabel('Autonomy Index Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend(title='Rater Type')
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
savefig(fig, os.path.join(OUT_DIR, "fig_autonomy_index_distribution.png"))

Loading data...
  Human raw: 174 responses, 51 vignettes
  LLM raw:   3,431 rows, 50 vignettes

Apples-to-apples matching...
✓ Matched & combined: 3,604 rows total (173 human, 3,431 LLM) across 50 vignettes

[Q1] Variance decomposition...
      Vignette=67.6%  Rater=15.1%
[Q2] Rater type effect (within-vignette FE)...
      β = +55.637  p = 0.0000
[Q3] Domain heterogeneity...
[Q4] Difficulty moderation...
      Interaction β = -0.3558  p = 0.0000
[Q5] ECI/SPI moderation...
[Q6] Per-model causal gap...

Generating causal figures:
  📊 /content/causal/figCC1_variance_rater.png
  📊 /content/causal/figCC2_domain_heterogeneity.png
  📊 /content/causal/figCC3_difficulty_moderation.png
  📊 /content/causal/figCC4_eci_spi_moderation.png
  📊 /content/causal/figCC5_per_model_forest.png
  📊 /content/causal/figCC6_dashboard.png

════════════════════════════════════════════════════════════════════════
  CAUSAL COMPARISON REPORT — HUMAN vs LLM AUTONOMY INDEX
════════════════════════════════════════════

### Explanation of Causal Analysis Results

Let's break down the results of the causal analysis step by step:

**Q1 — Variance Decomposition**
This analysis shows how much of the variation in the Autonomy Index (AI) score can be attributed to different factors:
*   **Vignette (case content): 67.6%** — This means that the specific scenario or case presented accounts for the largest portion of the differences in AI scores. In other words, the content of the vignettes heavily influences the autonomy ratings.
*   **Rater type (Human/LLM): 15.1%** — The type of rater (whether it's a human or an LLM) explains a significant, but smaller, portion of the AI score variance, indicating a systematic difference between how humans and LLMs rate autonomy.
*   **Interaction (case × rater): 2.3%** — This indicates a minor interaction effect, meaning that the difference between human and LLM ratings varies slightly depending on the specific vignette.
*   **Residual (unexplained): 15.0%** — This is the portion of variance not explained by the factors above, likely due to other unmeasured variables or random error.

**Q2 — Causal Effect of Rater Type (within-vignette)**
This result quantifies the direct causal impact of the rater type on the Autonomy Index, *holding the vignette content constant*.
*   **β = +55.637 (p < 0.001)** — This is a highly statistically significant finding. It means that, for the same vignette, **human raters score the Autonomy Index 55.64 points HIGHER than LLMs**.

**Q3 — Domain Heterogeneity (per-domain rater effect)**
This examines if the Human-LLM gap varies across the four clinical domains of autonomy:
*   **Value Awareness (VA): β = +49.88 (***)**
*   **Factual Understanding (FU): β = +60.80 (***)**
*   **Rational Deliberation (RD): β = +57.29 (***)**
*   **Intentional Action (IA): β = +54.57 (***)**

All four domains show a statistically significant positive beta, meaning that **human raters score all domains of autonomy significantly higher than LLMs**. The largest gap is in Factual Understanding (FU), followed closely by Rational Deliberation (RD) and Intentional Action (IA).

**Q4 — Difficulty Moderation (gap × vignette difficulty)**
This explores whether the Human-LLM gap changes depending on the difficulty of the vignette.
*   **Interaction β = -0.3558 (p < 0.001)** — This indicates a statistically significant negative interaction. It means that the **gap between human and LLM scores SIGNIFICANTLY narrows as vignette difficulty increases**. In other words, on harder cases, LLMs' scores get closer to human scores, or humans' scores decrease more than LLMs' on harder cases.

**Q5 — Contextual Modifiers (ECI, SPI)**
This checks if external constraints (ECI) or support provided (SPI) moderate the Human-LLM gap.
*   **ECI × is_human interaction: β = -0.0214 (p = 0.8244)** — Not statistically significant. This suggests that the presence or absence of external constraints does not significantly change the difference between human and LLM autonomy ratings.
*   **SPI × is_human interaction: β = -0.0021 (p = 0.9803)** — Not statistically significant. This indicates that the level of support provided also does not significantly moderate the Human-LLM gap.

**Q6 — Per-Model Causal Gap (LLM − Human, vignette-adjusted)**
This provides a breakdown of the causal gap for each individual LLM model compared to human raters, again, adjusted for vignette content. A negative value means the LLM scores lower than humans.
*   All listed LLMs show a statistically significant negative gap (indicated by `***`), meaning all of them score lower than human raters.
*   The gaps range from **-52.28 for Gemini 3.5** (the smallest gap, meaning it's closest to human ratings) to **-59.47 for Deepseek V4 Pro** (the largest gap). This shows that while all LLMs score lower, there's some variability in how far they deviate from human scores.

In summary, humans consistently score autonomy higher than LLMs across all domains, and this gap tends to narrow on more difficult vignettes.